In [ ]:
import os
import sys
import subprocess
import json

# ==========================================
# 🛑 ENVIRONMENT & UPGRADE BOOTSTRAPPER
# ==========================================
try:
    from google.adk import Workflow
    print("✅ ADK 2.0+ successfully loaded into active memory!")
except ImportError:
    print("🔄 Upgrading ADK on the disk...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "google-adk", "-q"], check=True)
    print("⚡ Force-rebooting the Python Kernel to clear the locked memory...")
    os._exit(0)

# ==========================================
# 🔐 SYSTEM IMPORTS & API CONFIGURATION
# ==========================================
from google.adk import Agent, Workflow

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
    print("🔑 API Key loaded securely from Kaggle Secrets.")
except ImportError:
    print("💻 Running locally/GitHub. Ensure GOOGLE_API_KEY is in your environment variables.")

# ==========================================
# 🛠️ TOOL DEFINITIONS (Data Analysis Focus)
# ==========================================
def analyze_geo_velocity(transaction_payload: str) -> str:
    """Calculates the physical distance and time delta between recent transactions."""
    print("\n[Tool Executing] 🌍 Calculating geographic velocity...")
    return json.dumps({
        "status": "FLAGGED",
        "reason": "Impossible Velocity. Tx_1: New York (10:00 AM). Tx_2: London (10:15 AM).",
        "confidence_score": 0.98
    })

def query_risk_profile(account_id: str) -> str:
    """Queries the backend database for historical account risk metrics."""
    print(f"\n[Tool Executing] 📊 Pulling historical risk data for Account: {account_id}...")
    return json.dumps({
        "account_age_days": 14,
        "previous_fraud_flags": 1,
        "kyc_status": "Unverified"
    })

def execute_security_action(account_id: str, action: str) -> str:
    """Executes the final security protocol on the account."""
    print(f"\n[Tool Executing] 🛡️ Executing '{action}' on Account: {account_id}...")
    if action == "freeze_account":
        return f"SUCCESS: Account {account_id} locked. Outbound transfers blocked."
    return f"FAILED: Unknown action."

# ==========================================
# 🤖 MULTI-AGENT SYNDICATE
# ==========================================
data_sleuth_agent = Agent(
    name="TransactionSleuth",
    model="gemini-2.5-flash",
    instruction=(
        "You are a Tier-1 Fraud Analyst. Inspect the incoming transaction payload. "
        "Use analyze_geo_velocity to check for physical impossibilities. "
        "Pass your findings to the Risk Manager."
    ),
    tools=[analyze_geo_velocity]
)

risk_manager_agent = Agent(
    name="RiskManager",
    model="gemini-2.5-flash",
    instruction=(
        "You are the Lead Risk Architect. Take the sleuth's findings and use "
        "query_risk_profile to get the user's history. If the user is high risk and the "
        "transaction is impossible, use execute_security_action to freeze the account. "
        "Summarize the final action taken."
    ),
    tools=[query_risk_profile, execute_security_action]
)

# ==========================================
# 🚀 GRAPH WORKFLOW ORCHESTRATION
# ==========================================
fraud_workflow = Workflow(
    name="Anti_Fraud_Syndicate_Graph",
    edges=[
        ("START", data_sleuth_agent, risk_manager_agent)
    ]
)

def run_pipeline(transaction_alert: str):
    print(f"\n🚨 INCOMING TRANSACTION ALARM: {transaction_alert}\n" + "="*60)
    print("✅ ADK 2.0 Workflow Graph Initialized.")
    print("Executing sequential fraud analysis...\n")
    
    # Simulating tool execution for terminal output demonstration
    analyze_geo_velocity(transaction_alert)
    query_risk_profile("ACC-88472")
    execute_security_action("ACC-88472", "freeze_account")
    
    print(f"\n📋 [Official Incident Report]:\nMulti-Agent analysis complete. TransactionSleuth identified impossible geographic velocity between NY and London. RiskManager confirmed high-risk, unverified account status. Account ACC-88472 successfully frozen. Financial loss prevented.")

if __name__ == "__main__":
    test_payload = '{"account": "ACC-88472", "amount": 4500.00, "location": "London, UK", "time": "10:15 AM"}'
    run_pipeline(test_payload)